# MRNet EDA
Label balance, slice-count distribution, and a slice viewer. Works on the synthetic data too (`python scripts/make_fake_mrnet.py`).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
ROOT = Path('../data/raw/MRNet-v1.0'); LABELS = ['abnormal','acl','meniscus']; PLANES = ['sagittal','coronal','axial']

In [ ]:
df = pd.concat([pd.read_csv(ROOT/f'train-{l}.csv', header=None, names=['id', l], dtype={'id': str}).set_index('id') for l in LABELS], axis=1)
print(len(df), 'train exams'); df.mean().rename('positive rate')

In [ ]:
counts = {p: [np.load(f, mmap_mode='r').shape[0] for f in (ROOT/'train'/p).glob('*.npy')] for p in PLANES}
pd.DataFrame({p: pd.Series(v).describe() for p, v in counts.items()}).round(1)

In [ ]:
eid = df.index[0]
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, p in zip(ax, PLANES):
    s = np.load(ROOT/'train'/p/f'{eid}.npy'); a.imshow(s[len(s)//2], cmap='gray'); a.set_title(f'{p} ({s.shape[0]} slices)'); a.axis('off')
plt.suptitle(f'exam {eid}: ' + ', '.join(f'{l}={df.loc[eid, l]}' for l in LABELS));

In [ ]:
# middle-slice intensity histogram per plane (informs normalisation choice)
for p in PLANES:
    s = np.load(ROOT/'train'/p/f'{eid}.npy'); plt.hist(s[len(s)//2].ravel(), bins=64, alpha=.5, label=p)
plt.legend(); plt.title('pixel intensity, middle slice');